In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
df = spark.readStream.table(f"{BRONZE}.sap_incremental")

def upsert_sap(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    microBatchDF.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.sap_asset_master")

df.writeStream.foreachBatch(upsert_sap).option("checkpointLocation", f"{CHECKPOINTS}/silver_sap").trigger(availableNow=True).start().awaitTermination()